In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd
import numpy as np

In [3]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Lam et al"

In [4]:
df_independent_2 = pd.read_excel(f"{path_input}/{name_source}/data.xlsx", sheet_name="independent_2")
df_independent_2["label"] = np.where(
    df_independent_2["SEQCLASS"] == "non-antioxidant",
    0,
    1
)

df_independent_2 = df_independent_2[["SEQUENCE", "label"]]
df_independent_2.rename(columns={"SEQUENCE": "sequence"}, inplace=True)
df_independent_2.head()

,sequence,label
0,MRSPSLAVAATTVLGLFSSSALAYYGNTTTVALTTTEFVTTCPYPT...,0
1,MRLRRLALFPGVALLLAAARLAAASDVLELTDDNFESRISDTGSAG...,0
2,MAPPQRHPQRSEQVLLLTLLGTLWGAAAAQIRYSIPEELEKGSFVG...,0
3,MSELSDIRREYTLGELHSEDVPNDPMDLFNAWLEVVRDSQIQDPTA...,0
4,MAGNAAVGVLALQGDVSEHISAFESAIQNLGLNIPVVPVRKAEQIL...,0


In [5]:
df_independent_1 = pd.read_excel(f"{path_input}/{name_source}/data.xlsx", sheet_name="independent_1", header=None)
df_independent_1.columns = ["sequence"]
df_independent_1.head()
df_independent_1['label'] = 1

In [6]:
df_training = pd.read_excel(f"{path_input}/{name_source}/data.xlsx", sheet_name="training", header=None)
df_training.columns = ["data_fasta"]
df_training.head()


,data_fasta
0,>anti_|1|training
1,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...
2,>anti_|1|training
3,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...
4,>anti_|1|training


In [7]:
df_training_data = []

for i in range(0, df_training.shape[0], 2):
    data_label = df_training["data_fasta"][i]
    
    if "non" in data_label:
        label=0
    else:
        label=1

    sequence = df_training["data_fasta"][i+1]

    row = {"sequence": sequence, "label": label}
    df_training_data.append(row)

df_training_data = pd.DataFrame(df_training_data)
df_training_data.head()

,sequence,label
0,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
1,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
2,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
3,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1
4,MANSGLWELITIGSAVRNVAKSYLKAEASSITAKQLYDASKITSSK...,1


- Concatenating data

In [8]:
df_concat = pd.concat([df_training_data, df_independent_1, df_independent_2], axis=0, ignore_index=True)

In [9]:
df_concat.shape, df_concat["sequence"].unique().shape

((2366, 2), (2322,))

- Working with duplicates

In [10]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_concat, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape

((44, 3), (0, 0), (2278, 2))

In [11]:
data_correct = pd.concat([df_consistent_duplicates, df_unique], axis=0, ignore_index=True)
data_correct = data_correct.drop(columns=["n_duplicates"])
data_correct.head()

,sequence,label
0,MAAHTILASAPSHTTFSLISPFSSTPTNALSSSLQSSSFNGLSFKL...,1
1,MAAICLPVAKHSFPSLLNTQTPKPLFSQNLHTIPLSSQSQICGLKF...,1
2,MAALDAIREALPEPARDIKLNLQAVLQPGTLTPAQRWGVAVATAAA...,1
3,MAFAVSTACRPSLLLPPRQRSSPPRPRPLLCTPSTAAFRRGALSAT...,1
4,MAFSVQMPALGESVTEGTVTRWLKQEGDTVELDEPLVEVSTDKVDT...,1


- Working with metadata

In [12]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
46,data.xlsx,Lam et al,Dataset,Static,Creative Commons Attribution License,No,2020,2020-10-06,2025-09-07,xlsx,"Sequence, UniProt ID",Enzyme/protein classification,Antioxidant,"Manually curated database, Sampling from UniPr...","Obtained from other databases, Sampling from U...",Supplementary Material,https://pmc.ncbi.nlm.nih.gov/articles/PMC75996...,No information


In [13]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'data.xlsx',
 'name source': 'Lam et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-06 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Manually curated database, Sampling from UniProt, Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Obtained from other databases, Sampling from UniProt, Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7599600/#_ad93_',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:30:59'}

In [14]:
dict_metadata['number_of_records'] = df_concat.shape[0]
dict_metadata['number_of_collected_sequences'] = df_concat.shape[0]
dict_metadata['number_of_unique_sequences'] = data_correct.shape[0]
dict_metadata['positive_examples'] = data_correct[data_correct["label"] == 1].shape[0]
dict_metadata['negative_examples'] = data_correct[data_correct["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': 'data.xlsx',
 'name source': 'Lam et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-06 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Manually curated database, Sampling from UniProt, Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Obtained from other databases, Sampling from UniProt, Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7599600/#_ad93_',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:30:59',
 'number_of_records': 2366,
 'number_of_collected_sequenc

- Exporting data

In [15]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
data_correct.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)